In [2]:
import torch
import os
import torch

from lxt.efficient import monkey_patch_zennit
from basemodel import load_timm_wrapper
from dino_patcher import DINOPatcher

In [3]:

monkey_patch_zennit(verbose=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model_wrapper, transforms, data_config = load_timm_wrapper(
        checkpoint_path="/workspaces/vast-gorilla/gorillawatch/models/ViTG-body_face-spac23+24v3-dedup.pth",
        backbone_name="vit_giant_patch14_dinov2.lvd142m",
        embedding_size=256,
        image_size=518,
        finetuned=True,
        device=DEVICE,
        bp_transforms=True,
        model_dtype=torch.float32,
    )

INFO:basemodel:Setting img_size to 518


Patched Zennit BasicHook's forward
Patched Zennit BasicHook's backward
--- Loading TimmWrapper Model ---
Building model architecture with backbone 'vit_giant_patch14_dinov2.lvd142m' and embedding size 256...


INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/vit_giant_patch14_dinov2.lvd142m)
INFO:timm.models._hub:[timm/vit_giant_patch14_dinov2.lvd142m] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:basemodel:No pooling layer reset necessary.


Using transforms as we did in the bachelor project
--- Model loading complete ---


In [4]:
for name, module in model_wrapper.named_modules():
    key = name
    print(f"{key:50} {module.__class__.__name__}")

                                                   TimmWrapper
model                                              VisionTransformer
model.patch_embed                                  PatchEmbed
model.patch_embed.proj                             Conv2d
model.patch_embed.norm                             Identity
model.pos_drop                                     Dropout
model.patch_drop                                   Identity
model.norm_pre                                     Identity
model.blocks                                       Sequential
model.blocks.0                                     Block
model.blocks.0.norm1                               LayerNorm
model.blocks.0.attn                                Attention
model.blocks.0.attn.qkv                            Linear
model.blocks.0.attn.q_norm                         Identity
model.blocks.0.attn.k_norm                         Identity
model.blocks.0.attn.attn_drop                      Dropout
model.blocks.0.attn.proj       

In [5]:
def inspect_hooks(model, label):
    print(f"\n--- {label} ---")
    for name, module in model.named_modules():
        if module._forward_hooks or module._backward_hooks:
            print(f"{name:50} {module.__class__.__name__}")
            print(f"   forward_hooks: {list(module._forward_hooks.keys())}")
            print(f"   backward_hooks: {list(module._backward_hooks.keys())}")

In [6]:
from lrp_helpers import create_vit_lrp_composite
import zennit.rules as z_rules
from zennit.composites import LayerMapComposite

zennit_comp_old = LayerMapComposite([
        (torch.nn.Conv2d, z_rules.Gamma(0.5)),
        (torch.nn.Linear, z_rules.Gamma(0.001)),
    ])
zennit_comp_new = create_vit_lrp_composite(conv_gamma=0.5, lin_gamma=0.001)

with DINOPatcher(model_wrapper):
    inspect_hooks(model_wrapper, "before")

    # Old comp
    zennit_comp_old.register(model_wrapper)
    inspect_hooks(model_wrapper, "old comp")
    zennit_comp_old.remove()

    # New comp
    zennit_comp_new.register(model_wrapper)
    inspect_hooks(model_wrapper, "new comp")
    for types, rule in zennit_comp_new.layer_map:
        print(types, rule)
    zennit_comp_new.remove()

--- Applying LRP patches ---

--- before ---

--- old comp ---
model.patch_embed.proj                             Conv2d
   forward_hooks: [1, 2]
   backward_hooks: []
model.blocks.0.attn.qkv                            Linear
   forward_hooks: [4, 5]
   backward_hooks: []
model.blocks.0.attn.proj                           Linear
   forward_hooks: [7, 8]
   backward_hooks: []
model.blocks.0.mlp.fc1                             Linear
   forward_hooks: [10, 11]
   backward_hooks: []
model.blocks.0.mlp.fc2                             Linear
   forward_hooks: [13, 14]
   backward_hooks: []
model.blocks.1.attn.qkv                            Linear
   forward_hooks: [16, 17]
   backward_hooks: []
model.blocks.1.attn.proj                           Linear
   forward_hooks: [19, 20]
   backward_hooks: []
model.blocks.1.mlp.fc1                             Linear
   forward_hooks: [22, 23]
   backward_hooks: []
model.blocks.1.mlp.fc2                             Linear
   forward_hooks: [25, 26]
  

In [1]:
from torchvision.models import vision_transformer

def get_vit_imagenet(device="cuda"):
    """
    Load a pre-trained Vision Transformer (ViT) model with ImageNet weights.

    Parameters:
    device (str): Device to load the model on ('cuda' or 'cpu')

    Returns:
    tuple: (model, weights) - The ViT model and its pre-trained weights
    """
    weights = vision_transformer.ViT_B_16_Weights.IMAGENET1K_V1
    model = vision_transformer.vit_b_16(weights=weights)
    model.eval()
    model.to(device)

    # Deactivate gradients on parameters to save memory
    for param in model.parameters():
        param.requires_grad = False

    return model, weights


# Load the pre-trained ViT model
model, weights = get_vit_imagenet()

In [8]:
from torch.nn import MultiheadAttention

print("-" * 80)
for name, module in model.named_modules():
    key = name if name else "model" # Handle the top-level module
    print(f"{key:50} {module.__class__.__name__}")

    # Add a special check for MultiheadAttention
    if isinstance(module, MultiheadAttention):
        print(f"{'':52}   Heads: {module.num_heads}, Embed Dim: {module.embed_dim}")
        # Print shape of the combined Q, K, V weight matrix
        print(f"{'':52}   In-Proj Shape: {module.in_proj_weight.shape}")
print("-" * 80)

--------------------------------------------------------------------------------
model                                              VisionTransformer
conv_proj                                          Conv2d
encoder                                            Encoder
encoder.dropout                                    Dropout
encoder.layers                                     Sequential
encoder.layers.encoder_layer_0                     EncoderBlock
encoder.layers.encoder_layer_0.ln_1                LayerNorm
encoder.layers.encoder_layer_0.self_attention      MultiheadAttention
                                                       Heads: 12, Embed Dim: 768
                                                       In-Proj Shape: torch.Size([2304, 768])
encoder.layers.encoder_layer_0.self_attention.out_proj NonDynamicallyQuantizableLinear
encoder.layers.encoder_layer_0.dropout             Dropout
encoder.layers.encoder_layer_0.ln_2                LayerNorm
encoder.layers.encoder_layer_0.mlp 

In [16]:
from torch import nn
from zennit.composites import NameLayerMapComposite
from zennit.rules import Epsilon, Gamma

def create_dinov2_lrp_composite(
    model: nn.Module,
    conv_gamma: float,
    ffn_gamma: float,
    epsilon: float = 1e-6,
    verbose: bool = False  # Add a verbose flag
):
    """
    Creates a Zennit composite for a ViT using NameLayerMapComposite.
    - Conv2d layers -> Gamma rule
    - Linear layers in FFN -> Gamma rule
    - Linear layers in Attention -> Epsilon rule
    """
    
    attn_linear_names = []
    ffn_linear_names = []
    conv_names = []
    
    # --- 1. Identify and categorize all relevant layers ---
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if '.attn.' in name:
                attn_linear_names.append(name)
            else:
                # This will include MLP layers and the final embedding layer if any
                ffn_linear_names.append(name)
        elif isinstance(module, nn.Conv2d):
            conv_names.append(name)

    if verbose:
        print("--- Zennit Composite Builder: Layer Identification ---")
        print(f"Found {len(conv_names)} Conv2d layers to be mapped to Gamma(gamma={conv_gamma})")
        # print("\n".join(f"  - {name}" for name in conv_names)) # Uncomment for full list
        
        print(f"\nFound {len(attn_linear_names)} Attention Linear layers to be mapped to Epsilon(epsilon={epsilon})")
        # Print the first few to confirm
        for name in attn_linear_names[:4]:
            print(f"  - {name}")
        if len(attn_linear_names) > 4:
            print("  - ...")

        print(f"\nFound {len(ffn_linear_names)} FFN/Other Linear layers to be mapped to Gamma(gamma={ffn_gamma})")
        for name in ffn_linear_names[:4]:
            print(f"  - {name}")
        if len(ffn_linear_names) > 4:
            print("  - ...")
        print("----------------------------------------------------\n")

    # --- 2. Define the specific name-based overrides for attention layers ---
    name_map = [
        (attn_linear_names, Epsilon(epsilon=epsilon)),
    ]

    # --- 3. Define the general, type-based rules for all other layers ---
    layer_map = [
        (nn.Conv2d, Gamma(gamma=conv_gamma)),
        (nn.Linear, Gamma(gamma=ffn_gamma)), # This is the default for any Linear not in name_map
    ]

    return NameLayerMapComposite(name_map, layer_map)

# --- Example Usage ---
# Assuming 'model_wrapper' is your model instance
CONV_GAMMA = 0.5
FFN_GAMMA = 0.001
EPSILON = 1e-6

vit_composite = create_dinov2_lrp_composite(
    model_wrapper,
    conv_gamma=CONV_GAMMA,
    ffn_gamma=FFN_GAMMA,
    epsilon=EPSILON,
    verbose=True
)




def print_registered_rules(model: nn.Module):
    '''
    Iterates through a model after hooks have been registered by a Zennit attributor
    and prints the specific rule attached to each relevant layer.
    '''
    print("\n--- Verifying Rules Registered on Model Layers ---")
    found_hooks = False
    for name, module in model.named_modules():
        # Zennit attaches the hook context to the '_zennit_hook' attribute
        if hasattr(module, '_zennit_hook'):
            found_hooks = True
            rule = module._zennit_hook.rule
            print(f"Layer: {name:<40} Type: {module.__class__.__name__:<10} --> Rule: {rule}")
            
    if not found_hooks:
        print("No Zennit hooks found. Did you create the LRP attributor object yet?")

vit_composite.register(model_wrapper)
print_registered_rules(model_wrapper)

--- Zennit Composite Builder: Layer Identification ---
Found 1 Conv2d layers to be mapped to Gamma(gamma=0.5)

Found 80 Attention Linear layers to be mapped to Epsilon(epsilon=1e-06)
  - model.blocks.0.attn.qkv
  - model.blocks.0.attn.proj
  - model.blocks.1.attn.qkv
  - model.blocks.1.attn.proj
  - ...

Found 81 FFN/Other Linear layers to be mapped to Gamma(gamma=0.001)
  - model.blocks.0.mlp.fc1
  - model.blocks.0.mlp.fc2
  - model.blocks.1.mlp.fc1
  - model.blocks.1.mlp.fc2
  - ...
----------------------------------------------------


--- Verifying Rules Registered on Model Layers ---
No Zennit hooks found. Did you create the LRP attributor object yet?


In [18]:
from zennit.composites import NameLayerMapComposite
from zennit.rules import Epsilon, Gamma
from torch import nn

# --- Step 1: Create a Verbose Composite Class ---
class VerboseNameLayerMapComposite(NameLayerMapComposite):
    '''
    An extension of NameLayerMapComposite that prints the rule being
    applied to each named module during registration.
    '''
    def _apply(self, module, name=''):
        # This is the core method where the composite decides which rule to use.
        # We will hook into it to print the decision.
        
        # First, let the parent class find the appropriate rule.
        rule = super()._map(module, name)

        # If a rule was found (i.e., it's not None), print our verification message.
        if rule is not None:
            # We check the type of rule to provide a more informative message.
            if isinstance(rule, Epsilon):
                rule_info = f"Epsilon(epsilon={rule.epsilon})"
            elif isinstance(rule, Gamma):
                rule_info = f"Gamma(gamma={rule.gamma})"
            else:
                rule_info = str(rule)
            
            print(f"[Zennit VERIFY] Module '{name}' --> Rule: {rule_info}")
        
        # Crucially, call the original _apply method to actually register the hooks.
        return super()._apply(module, name)

# --- Step 2: Use this new composite in your creator function ---
def create_dinov2_lrp_composite_with_verification(
    model: nn.Module,
    conv_gamma: float,
    ffn_gamma: float,
    epsilon: float = 1e-6
):
    """
    Creates the Zennit composite using our new Verbose class for verification.
    """
    attn_linear_names = [
        name for name, mod in model.named_modules() 
        if isinstance(mod, nn.Linear) and '.attn.' in name
    ]
    
    name_map = [
        (attn_linear_names, Epsilon(epsilon=epsilon)),
    ]

    layer_map = [
        (nn.Conv2d, Gamma(gamma=conv_gamma)),
        (nn.Linear, Gamma(gamma=ffn_gamma)),
    ]

    # Use the Verbose composite instead of the standard one
    return VerboseNameLayerMapComposite(name_map, layer_map)


zennit_comp = create_dinov2_lrp_composite_with_verification(
        model_wrapper,
        conv_gamma=CONV_GAMMA,
        ffn_gamma=FFN_GAMMA,
        epsilon=EPSILON
    )
    

print("\n--- Verifying Zennit Rule Registration ---")
zennit_comp.register(model_wrapper)

test_tensor = torch.randn(1, 3, 518, 518).to(DEVICE)
test_tensor.grad = None
test_embed = model_wrapper(test_tensor.requires_grad_())


--- Verifying Zennit Rule Registration ---


In [26]:
from torch import nn
from zennit.composites import NameLayerMapComposite
from zennit.rules import Epsilon, Gamma, Pass
from zennit.layer import Sum

def create_dinov2_lrp_composite(
    model: nn.Module,
    conv_gamma: float,
    ffn_gamma: float,
    epsilon: float = 1e-6,
    verbose: bool = False
):
    """
    Creates a Zennit composite for a ViT using NameLayerMapComposite.
    - Conv2d layers -> Gamma(conv_gamma)
    - Linear layers in FFN -> Gamma(ffn_gamma)
    - Linear layers in Attention -> Epsilon(epsilon)
    """
    if verbose:
        print("--- Building DINOv2 LRP Composite ---")

    # 1. Find the names of all Linear layers
    all_linear_names = {name for name, mod in model.named_modules() if isinstance(mod, nn.Linear)}
    
    # 2. Find the names of attention-specific linear layers to override
    attn_linear_names = {name for name in all_linear_names if '.attn.' in name}
    
    # 3. The rest are FFN linear layers
    ffn_linear_names = all_linear_names - attn_linear_names

    if verbose:
        print(f"Found {len(attn_linear_names)} attention Linear layers to be mapped to Epsilon.")
        print("  - Sample:", list(attn_linear_names)[:5])
        print(f"Found {len(ffn_linear_names)} FFN/other Linear layers to be mapped to Gamma.")
        print("  - Sample:", list(ffn_linear_names)[:5])

    # 4. Define the map of specific name-based overrides.
    # The format is a list of (list_of_names, rule).
    name_map = [
        (list(attn_linear_names), Epsilon(epsilon=epsilon)),
    ]

    # 5. Define the map of general, type-based rules for everything else.
    layer_map = [
        (nn.Conv2d, Gamma(gamma=conv_gamma)),
        (nn.Linear, Gamma(gamma=ffn_gamma)), # This is the default for non-attention Linear layers
    ]
    
    # For ViTs, we often need to handle skip connections (Sum layers) and LayerNorm.
    # A more complete composite includes rules for these.
    # The 'Pass' rule propagates relevance without modification.
    full_layer_map = layer_map + [
        (nn.LayerNorm, Pass()),
        (nn.Dropout, Pass()),
        (Sum, Pass()),
    ]
    
    if verbose:
        print("---------------------------------------")
        
    # NameLayerMapComposite checks name_map first, then layer_map.
    return NameLayerMapComposite(name_map, full_layer_map)

# --- Usage for Sanity Check ---
# Just instantiate it with verbose=True to see the output
# No need to run the model yet.
print("Level 1: Static Verification")
vit_composite = create_dinov2_lrp_composite(model_wrapper, 0.5, 0.1, verbose=True)

Level 1: Static Verification
--- Building DINOv2 LRP Composite ---
Found 80 attention Linear layers to be mapped to Epsilon.
  - Sample: ['model.blocks.36.attn.qkv', 'model.blocks.0.attn.proj', 'model.blocks.10.attn.qkv', 'model.blocks.20.attn.proj', 'model.blocks.31.attn.proj']
Found 81 FFN/other Linear layers to be mapped to Gamma.
  - Sample: ['model.blocks.38.mlp.fc2', 'model.blocks.30.mlp.fc1', 'model.blocks.32.mlp.fc2', 'model.blocks.36.mlp.fc2', 'model.blocks.19.mlp.fc2']
---------------------------------------


In [30]:
from zennit.composites import LayerMapComposite

def test_composite_specificity(
    model_wrapper: nn.Module,
):
    """
    Definitive test to confirm the NameLayerMapComposite is working.
    Compares the intended composite (Attn->Epsilon, FFN->Gamma) against
    a control composite where ALL linear layers get the Gamma rule.
    If the relevance maps are different, the name-based override is working.
    """
    # Use a smaller, faster input for testing
    input_tensor = torch.randn(1, 3, 518, 518).to(DEVICE)
    model_wrapper.to(DEVICE).eval()

    # --- Configuration ---
    CONV_GAMMA = 0.5
    FFN_GAMMA = 0.1
    EPSILON = 1e-6

    def get_relevance(composite: NameLayerMapComposite) -> torch.Tensor:
        """Helper to run a single LRP pass and return relevance."""
        # Ensure model is clean of hooks
        composite.remove() 
        
        # Zero out gradients
        if input_tensor.grad is not None:
            input_tensor.grad.detach_()
            input_tensor.grad.zero_()
        input_tensor.requires_grad_(True)
        
        try:
            composite.register(model_wrapper)
            
            # Forward pass
            emb = model_wrapper(input_tensor)
            
            # Create a simple score for backpropagation
            # Using the element with the highest absolute value is a stable choice
            score = emb.flatten()[emb.abs().argmax()]
            
            # Backward pass (LRP)
            score.backward()
            
            if input_tensor.grad is None:
                relevance = torch.zeros_like(input_tensor)
            else:
                relevance = (input_tensor * input_tensor.grad)
                
        finally:
            composite.remove() # Crucial to clean up hooks
            
        return relevance.detach().cpu()

    # --- Create Composite A (Your intended, specific composite) ---
    print("Creating Composite A: Attn -> Epsilon, FFN -> Gamma")
    composite_A = create_dinov2_lrp_composite(
        model_wrapper,
        conv_gamma=CONV_GAMMA,
        ffn_gamma=FFN_GAMMA,
        epsilon=EPSILON,
        verbose=True
    )
    
    # --- Create Composite B (Control: all Linear layers use Gamma) ---
    # We achieve this by simply using a standard LayerMapComposite without name overrides.
    print("\nCreating Composite B: All Linear -> Gamma (Control)")
    control_layer_map = [
        (nn.Conv2d, Gamma(gamma=CONV_GAMMA)),
        (nn.Linear, Gamma(gamma=FFN_GAMMA)), # All linear layers get this rule
        (nn.LayerNorm, Pass()),
        (nn.Dropout, Pass()),
        (Sum, Pass()),
    ]
    composite_B = LayerMapComposite(control_layer_map)


    # --- Run LRP for both ---
    print("\nRunning LRP with Composite A...")
    relevance_A = get_relevance(composite_A)
    
    print("Running LRP with Composite B...")
    relevance_B = get_relevance(composite_B)

    # --- Compare the results ---
    difference = (relevance_A - relevance_B).abs()
    max_diff = difference.max().item()
    mean_diff = difference.mean().item()

    print("\n--- Test Results ---")
    print(f"Max absolute difference between relevance maps: {max_diff:.8f}")
    print(f"Mean absolute difference between relevance maps: {mean_diff:.8f}")

    if max_diff > 1e-9: # Use a small threshold to account for float precision
        print("\n✅ SUCCESS: The relevance maps are different.")
        print("This confirms that the name-based override for attention layers is active and applying a different rule.")
    else:
        print("\n❌ FAILURE: The relevance maps are identical.")
        print("This indicates the name-based override had no effect.")
        
    return max_diff, mean_diff

test_composite_specificity(model_wrapper)

Creating Composite A: Attn -> Epsilon, FFN -> Gamma
--- Building DINOv2 LRP Composite ---
Found 80 attention Linear layers to be mapped to Epsilon.
  - Sample: ['model.blocks.36.attn.qkv', 'model.blocks.0.attn.proj', 'model.blocks.10.attn.qkv', 'model.blocks.20.attn.proj', 'model.blocks.31.attn.proj']
Found 81 FFN/other Linear layers to be mapped to Gamma.
  - Sample: ['model.blocks.38.mlp.fc2', 'model.blocks.30.mlp.fc1', 'model.blocks.32.mlp.fc2', 'model.blocks.36.mlp.fc2', 'model.blocks.19.mlp.fc2']
---------------------------------------

Creating Composite B: All Linear -> Gamma (Control)

Running LRP with Composite A...
Running LRP with Composite B...

--- Test Results ---
Max absolute difference between relevance maps: 0.44482422
Mean absolute difference between relevance maps: 0.00030811

✅ SUCCESS: The relevance maps are different.
This confirms that the name-based override for attention layers is active and applying a different rule.


(0.44482421875, 0.0003081067115999758)

In [23]:
import torch
from torch import nn
from zennit.composites import NameLayerMapComposite
from zennit.rules import Epsilon, Gamma

def create_dinov2_lrp_composite(model: nn.Module, conv_gamma: float, ffn_gamma: float, epsilon: float = 1e-6):
    # Sammle alle nn.Linear-Namen, die 'attn' im hierarchischen Namen haben.
    attn_linear_names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and ('attn' in name):
            attn_linear_names.append(name)

    print(f"[DEBUG] Found {len(attn_linear_names)} attention Linear modules (sample up to 10):")
    for n in attn_linear_names[:10]:
        print("  -", n)

    # Name map: specific overrides for attention linears
    name_map = [(attn_linear_names, Epsilon(epsilon=epsilon))] if attn_linear_names else []

    # Layer map: general rules
    layer_map = [
        (nn.Conv2d, Gamma(gamma=conv_gamma)),
        (nn.Linear, Gamma(gamma=ffn_gamma)),
    ]
    comp = NameLayerMapComposite(name_map, layer_map)
    # Optional: expose map for debugging
    comp._debug_attn_names = attn_linear_names
    return comp

# --- Test wrapper that checks effect of lin_gamma ---
def test_lrp_gamma_effect(
    model_wrapper: nn.Module,
    input_tensor: torch.Tensor,
    conv_gamma=0.5,
    ffn_gamma_low=0.0,
    ffn_gamma_high=1.0,
    epsilon=1e-6,
    device='cpu'
):
    model_wrapper.to(device)
    model_wrapper.eval()
    input_tensor = input_tensor.to(device)

    # Sammle attn_names ein Mal direkt
    tmp_comp = create_dinov2_lrp_composite(model_wrapper, conv_gamma, ffn_gamma_low, epsilon)
    attn_names = getattr(tmp_comp, '_debug_attn_names', [])
    # remove gleich wieder, weil tmp_comp nur zum Sammeln diente
    tmp_comp.remove()

    def run_with_gamma(ffn_gamma):
        comp = create_dinov2_lrp_composite(model_wrapper, conv_gamma=conv_gamma, ffn_gamma=ffn_gamma, epsilon=epsilon)
        comp.register(model_wrapper)
        try:
            if input_tensor.grad is not None:
                input_tensor.grad.detach_()
                input_tensor.grad.zero_()
            input_tensor.requires_grad_()

            emb = model_wrapper(input_tensor)
            score = emb.flatten().abs().sum()
            score.backward()

            if input_tensor.grad is None:
                relevance = torch.zeros_like(input_tensor.sum(1, keepdim=True))
            else:
                relevance = (input_tensor * input_tensor.grad).sum(1, keepdim=True)
        finally:
            comp.remove()
        return relevance.detach().cpu()

    rel_low = run_with_gamma(ffn_gamma_low)
    rel_high = run_with_gamma(ffn_gamma_high)
    diff = (rel_high - rel_low).abs()

    return {
        'rel_low': rel_low,
        'rel_high': rel_high,
        'diff': diff,
        'max_diff': diff.max().item(),
        'mean_diff': diff.mean().item(),
        'attn_names': attn_names,
    }

# --- Example usage ---
dummy_input = torch.randn(1, 3, 518, 518).to(DEVICE)
stats = test_lrp_gamma_effect(model_wrapper, dummy_input, conv_gamma=0.5, ffn_gamma_low=0.0, ffn_gamma_high=0.5)


[DEBUG] Found 80 attention Linear modules (sample up to 10):
  - model.blocks.0.attn.qkv
  - model.blocks.0.attn.proj
  - model.blocks.1.attn.qkv
  - model.blocks.1.attn.proj
  - model.blocks.2.attn.qkv
  - model.blocks.2.attn.proj
  - model.blocks.3.attn.qkv
  - model.blocks.3.attn.proj
  - model.blocks.4.attn.qkv
  - model.blocks.4.attn.proj
[DEBUG] Found 80 attention Linear modules (sample up to 10):
  - model.blocks.0.attn.qkv
  - model.blocks.0.attn.proj
  - model.blocks.1.attn.qkv
  - model.blocks.1.attn.proj
  - model.blocks.2.attn.qkv
  - model.blocks.2.attn.proj
  - model.blocks.3.attn.qkv
  - model.blocks.3.attn.proj
  - model.blocks.4.attn.qkv
  - model.blocks.4.attn.proj


KeyboardInterrupt: 